In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

print("Project directory:")
print(PROJECT_DIR)

Project directory:
/content/drive/MyDrive/MTechIndProj/MoM_Project


In [3]:
required_folders = [
    "01_audio_vad",
    "02_asr_diarization",
    "03_dialogue_act",
    "04_mom_generation",
    "05_evidence_retrieval",
    "06_verification",
    "07_dashboard",
    "08_evaluation",
    "data/raw",
    "data/audio",
    "data/vad",
    "data/transcripts",
    "data/dialogue_acts",
    "data/references",
    "models/asr",
    "models/diarization",
    "models/dialogue_act",
    "models/llm",
    "outputs/mom",
    "outputs/evidence",
    "outputs/verified",
    "outputs/pdf",
    "evaluation_results"
]

for folder in required_folders:
    path = os.path.join(PROJECT_DIR, folder)

    if os.path.exists(path):
        print("✓", folder)
    else:
        print("✗ MISSING:", folder)

✓ 01_audio_vad
✓ 02_asr_diarization
✓ 03_dialogue_act
✓ 04_mom_generation
✓ 05_evidence_retrieval
✓ 06_verification
✓ 07_dashboard
✓ 08_evaluation
✓ data/raw
✓ data/audio
✓ data/vad
✓ data/transcripts
✓ data/dialogue_acts
✓ data/references
✓ models/asr
✓ models/diarization
✓ models/dialogue_act
✓ models/llm
✓ outputs/mom
✓ outputs/evidence
✓ outputs/verified
✓ outputs/pdf
✓ evaluation_results


In [4]:
from datasets import load_dataset

print("Datasets library is ready.")

Datasets library is ready.


In [5]:
from datasets import load_dataset

meetingbank = load_dataset("huuuyeah/meetingbank")

print(meetingbank)

README.md:   0%|          | 0.00/3.32k [00:00<?, ?B/s]

train.json: reconstructing file:   0%|          |  0.00B / 88.4MB            

train.json: downloading bytes:           |  0.00B            

validation.json: reconstructing file:   0%|          |  0.00B / 13.2MB            

validation.json: downloading bytes:           |  0.00B            

...

test.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/861 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/862 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 5169
    })
    validation: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 861
    })
    test: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 862
    })
})


In [6]:
print("Dataset splits:")

for split in meetingbank:
    print(
        split,
        "→",
        len(meetingbank[split]),
        "records"
    )

Dataset splits:
train → 5169 records
validation → 861 records
test → 862 records


In [7]:
sample = meetingbank["train"][0]

print("Available fields:")
print(sample.keys())

Available fields:
dict_keys(['summary', 'uid', 'id', 'transcript'])


In [8]:
print("\nMeeting ID:")
print(sample["id"])

print("\nUID:")
print(sample["uid"])


Meeting ID:
0

UID:
DenverCityCouncil_05012017_17-0161


In [9]:
print("\nTranscript:")
print(sample["transcript"][:3000])


Transcript:
Please refrain from profane or obscene speech. Direct your comments to council as a whole and refrain from individual or personal attacks. Councilwoman Gilmore, will you please put Council Bill 161 on the floor? Yes, President Brooks, I move that council bill 161 as amended, be placed upon final consideration and do pass. It has been moved and seconded. Councilman. Clerk, go ahead and offer your motion to further amend. Thank you, Mr. President. I move that council bill 161 as amended be further amended in the following particulars on page five, line one, strike 13 .1.9 and replace with 13 .1. ten on page five. Line three strike 13 1.9.3 and replace with 13 one dot and four. Add a new section to the bill which reads as follows Section six A except as otherwise provided in Section six B of this ordinance. With respect to certain site development plan applications, the amendments to the Denver Zoning Code adopted by this ordinance take effect on May five, 2017. B Notwithstan

In [10]:
print("\nReference summary:")
print(sample["summary"][:2000])


Reference summary:
AS AMENDED a bill for an ordinance amending the Denver Zoning Code to revise parking exemptions for pre-existing small zone lots. Approves a text amendment to the Denver Zoning Code to revise the Pre-Existing Small Zone Lot parking exemption. The Committee approved filing this bill at its meeting on 2-14-17. On 2-27-17, Council held this item in Committee to 3-20-17. Amended 3-20-17 to ensure that the parking exemption is applied for all uses. Some parking requirements are calculated based on gross floor area while others are on number of units and not explicitly for gross floor area, to further clarify the legislative intent of the proposed bill to emphasize the city’s commitment to more comprehensively address transportation demand management strategies in the short term, and to require a Zoning Permit with Informational Notice for all new buildings on Pre-Existing Small Zone Lots that request to use the small lot parking exemption; Enables all expansions to exist

In [11]:
meeting_id = str(sample["id"])

reference_transcript_path = os.path.join(
    PROJECT_DIR,
    "data",
    "references",
    f"{meeting_id}_reference_transcript.txt"
)

with open(
    reference_transcript_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(sample["transcript"])

print("Saved:")
print(reference_transcript_path)

Saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/references/0_reference_transcript.txt


In [12]:
reference_summary_path = os.path.join(
    PROJECT_DIR,
    "data",
    "references",
    f"{meeting_id}_reference_summary.txt"
)

with open(
    reference_summary_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(sample["summary"])

print("Saved:")
print(reference_summary_path)

Saved:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/references/0_reference_summary.txt


In [13]:
import pandas as pd

metadata = pd.DataFrame([
    {
        "meeting_id": meeting_id,
        "uid": sample["uid"],
        "split": "train",
        "media_path": "",
        "reference_transcript": reference_transcript_path,
        "reference_summary": reference_summary_path
    }
])

metadata_path = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "meeting_metadata.csv"
)

metadata.to_csv(
    metadata_path,
    index=False
)

print("Metadata saved at:")
print(metadata_path)

display(metadata)

Metadata saved at:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/meeting_metadata.csv


,meeting_id,uid,split,media_path,reference_transcript,reference_summary
0,0,DenverCityCouncil_05012017_17-0161,train,,/content/drive/MyDrive/MTechIndProj/MoM_Projec...,/content/drive/MyDrive/MTechIndProj/MoM_Projec...


In [14]:
print(meetingbank)
print(sample.keys())
print(sample["uid"])

DatasetDict({
    train: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 5169
    })
    validation: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 861
    })
    test: Dataset({
        features: ['summary', 'uid', 'id', 'transcript'],
        num_rows: 862
    })
})
dict_keys(['summary', 'uid', 'id', 'transcript'])
DenverCityCouncil_05012017_17-0161


✅ MeetingBank is loaded correctly.
That 6,892 total records is consistent with the segment-level MeetingBank benchmark we discussed.

Your selected first record is:
UID:
DenverCityCouncil_05012017_17-0161

Now we will try to find its matching audio from UID

In [15]:
print("Meeting ID:", sample["id"])
print("UID:", sample["uid"])

print("\nReference transcript:")
print(reference_transcript_path)

print("\nReference summary:")
print(reference_summary_path)

print("\nMetadata:")
print(metadata_path)

Meeting ID: 0
UID: DenverCityCouncil_05012017_17-0161

Reference transcript:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/references/0_reference_transcript.txt

Reference summary:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/references/0_reference_summary.txt

Metadata:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/meeting_metadata.csv


In [16]:
# Display the beginning of the reference transcript

# This is useful because later we can manually verify the ASR output.

with open(
    reference_transcript_path,
    "r",
    encoding="utf-8"
) as f:
    reference_text = f.read()

print(reference_text[:5000])

Please refrain from profane or obscene speech. Direct your comments to council as a whole and refrain from individual or personal attacks. Councilwoman Gilmore, will you please put Council Bill 161 on the floor? Yes, President Brooks, I move that council bill 161 as amended, be placed upon final consideration and do pass. It has been moved and seconded. Councilman. Clerk, go ahead and offer your motion to further amend. Thank you, Mr. President. I move that council bill 161 as amended be further amended in the following particulars on page five, line one, strike 13 .1.9 and replace with 13 .1. ten on page five. Line three strike 13 1.9.3 and replace with 13 one dot and four. Add a new section to the bill which reads as follows Section six A except as otherwise provided in Section six B of this ordinance. With respect to certain site development plan applications, the amendments to the Denver Zoning Code adopted by this ordinance take effect on May five, 2017. B Notwithstanding Section 

In [17]:
print(sample["uid"])
print(sample["id"])
print(reference_transcript_path)
print(reference_summary_path)

DenverCityCouncil_05012017_17-0161
0
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/references/0_reference_transcript.txt
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/references/0_reference_summary.txt


In [18]:
!pip install -q huggingface_hub

In [19]:
from huggingface_hub import list_repo_files

audio_files = list_repo_files(
    repo_id="huuuyeah/MeetingBank_Audio",
    repo_type="dataset"
)

print("Number of files:", len(audio_files))
print("\nFirst 30 files:")
for f in audio_files[:30]:
    print(f)

Number of files: 85

First 30 files:
.gitattributes
Alameda/.DS_Store
Alameda/Alameda-transcripts-videolist.zip
Alameda/mp3/alameda-1.zip
Alameda/mp3/alameda-2.zip
Alameda/mp3/alameda-3.zip
Alameda/mp3/alameda-4.zip
Alameda/mp3/alameda-5.zip
Alameda/mp3/alameda-6.zip
Alameda/mp3/alameda-7.zip
Alameda/mp3/alameda-8.zip
Alameda/mp3/alameda-9.zip
Boston/boston_video_list.txt
Boston/mp3/Boston.zip
Boston/transcripts/transcripts.zip
Denver/Denver-transcripts-videolist.zip
Denver/mp3/Denver-1.zip
Denver/mp3/Denver-10.zip
Denver/mp3/Denver-11.zip
Denver/mp3/Denver-12.zip
Denver/mp3/Denver-13.zip
Denver/mp3/Denver-14.zip
Denver/mp3/Denver-15.zip
Denver/mp3/Denver-16.zip
Denver/mp3/Denver-17.zip
Denver/mp3/Denver-18.zip
Denver/mp3/Denver-19.zip
Denver/mp3/Denver-2.zip
Denver/mp3/Denver-20.zip
Denver/mp3/Denver-21.zip


In [20]:
MEETING_UID = "DenverCityCouncil_05012017_17-0161"

matching_files = [
    f for f in audio_files
    if MEETING_UID in f
]

print("Matching files:")
for f in matching_files:
    print(f)

Matching files:


In [21]:
denver_files = [
    f for f in audio_files
    if "DenverCityCouncil" in f
]

print("Denver files:", len(denver_files))

for f in denver_files[:50]:
    print(f)

Denver files: 0


In [22]:
matching_files = [
    f for f in audio_files
    if "DenverCityCouncil_05012017_17-0161" in f
]

print(matching_files)

[]


In [23]:
MEETING_ID = "DenverCityCouncil_05012017"

print("Full Meeting ID:")
print(MEETING_ID)

Full Meeting ID:
DenverCityCouncil_05012017


In [24]:
import requests
import json
import os

metadata_url = "https://raw.githubusercontent.com/YebowenHu/MeetingBank-utils/main/MeetingBank.json"

response = requests.get(metadata_url)

print("Status code:", response.status_code)
print("Downloaded bytes:", len(response.content))

Status code: 404
Downloaded bytes: 14


In [25]:
import os

bad_metadata_file = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "MeetingBank.json"
)

if os.path.exists(bad_metadata_file):
    os.remove(bad_metadata_file)
    print("Removed incorrect metadata file.")
else:
    print("No incorrect metadata file found.")

No incorrect metadata file found.


In [26]:
from huggingface_hub import list_repo_files

audio_files = list_repo_files(
    repo_id="huuuyeah/MeetingBank_Audio",
    repo_type="dataset"
)

print("Total repository entries:", len(audio_files))

print("\nFirst 50 entries:")
for file in audio_files[:50]:
    print(file)

Total repository entries: 85

First 50 entries:
.gitattributes
Alameda/.DS_Store
Alameda/Alameda-transcripts-videolist.zip
Alameda/mp3/alameda-1.zip
Alameda/mp3/alameda-2.zip
Alameda/mp3/alameda-3.zip
Alameda/mp3/alameda-4.zip
Alameda/mp3/alameda-5.zip
Alameda/mp3/alameda-6.zip
Alameda/mp3/alameda-7.zip
Alameda/mp3/alameda-8.zip
Alameda/mp3/alameda-9.zip
Boston/boston_video_list.txt
Boston/mp3/Boston.zip
Boston/transcripts/transcripts.zip
Denver/Denver-transcripts-videolist.zip
Denver/mp3/Denver-1.zip
Denver/mp3/Denver-10.zip
Denver/mp3/Denver-11.zip
Denver/mp3/Denver-12.zip
Denver/mp3/Denver-13.zip
Denver/mp3/Denver-14.zip
Denver/mp3/Denver-15.zip
Denver/mp3/Denver-16.zip
Denver/mp3/Denver-17.zip
Denver/mp3/Denver-18.zip
Denver/mp3/Denver-19.zip
Denver/mp3/Denver-2.zip
Denver/mp3/Denver-20.zip
Denver/mp3/Denver-21.zip
Denver/mp3/Denver-3.zip
Denver/mp3/Denver-4.zip
Denver/mp3/Denver-5.zip
Denver/mp3/Denver-6.zip
Denver/mp3/Denver-7.zip
Denver/mp3/Denver-8.zip
Denver/mp3/Denver-9.zip
K

In [27]:
denver_files = [
    f for f in audio_files
    if f.lower().startswith("denver/")
]

print("Number of Denver entries:", len(denver_files))

for f in denver_files[:100]:
    print(f)

Number of Denver entries: 22
Denver/Denver-transcripts-videolist.zip
Denver/mp3/Denver-1.zip
Denver/mp3/Denver-10.zip
Denver/mp3/Denver-11.zip
Denver/mp3/Denver-12.zip
Denver/mp3/Denver-13.zip
Denver/mp3/Denver-14.zip
Denver/mp3/Denver-15.zip
Denver/mp3/Denver-16.zip
Denver/mp3/Denver-17.zip
Denver/mp3/Denver-18.zip
Denver/mp3/Denver-19.zip
Denver/mp3/Denver-2.zip
Denver/mp3/Denver-20.zip
Denver/mp3/Denver-21.zip
Denver/mp3/Denver-3.zip
Denver/mp3/Denver-4.zip
Denver/mp3/Denver-5.zip
Denver/mp3/Denver-6.zip
Denver/mp3/Denver-7.zip
Denver/mp3/Denver-8.zip
Denver/mp3/Denver-9.zip


In [28]:
from huggingface_hub import hf_hub_download

denver_map_path = hf_hub_download(
    repo_id="huuuyeah/MeetingBank_Audio",
    filename="Denver/Denver-transcripts-videolist.zip",
    repo_type="dataset"
)

print("Downloaded mapping file:")
print(denver_map_path)

Denver/Denver-transcripts-videolist.zip: reconstructing file:   0%|          |  0.00B /  681MB            

Denver/Denver-transcripts-videolist.zip: downloading bytes:           |  0.00B            

Downloaded mapping file:
/root/.cache/huggingface/hub/datasets--huuuyeah--MeetingBank_Audio/snapshots/27779a666ff5fd879f4c5567489ff47e82364abd/Denver/Denver-transcripts-videolist.zip


In [29]:
import zipfile

with zipfile.ZipFile(denver_map_path, "r") as z:
    files_inside = z.namelist()

print("Files inside ZIP:")
for f in files_inside:
    print(f)

Files inside ZIP:
denver_video_list.txt
transcripts/denver_0092aecb-4403-44ad-b9e4-a7a2733682af.mp3.transcript.json
transcripts/denver_009b1463-baf3-42f8-b3de-2820feea767c.mp3.transcript.json
transcripts/denver_01305102-03f9-4709-879f-40045397a390.mp3.transcript.json
transcripts/denver_0141a1aa-135d-41e6-a4c0-a97a6d93b717.mp3.transcript.json
transcripts/denver_05a9ae73-c724-4be9-8823-321b6ae9ee5d.mp3.transcript.json
transcripts/denver_05beac5d-717d-4b72-8d56-14b0c7909150.mp3.transcript.json
transcripts/denver_06505c2e-281d-462e-939c-ccb661ded293.mp3.transcript.json
transcripts/denver_0715097f-6484-4226-9433-ffae62bdf03e.mp3.transcript.json
transcripts/denver_07708b79-3284-474a-97c4-f4b1e566fbb7.mp3.transcript.json
transcripts/denver_077d2891-3e7c-4009-9cbb-7780290a5215.mp3.transcript.json
transcripts/denver_07a003c5-1b87-4ce9-a118-92c6d272f297.mp3.transcript.json
transcripts/denver_07c1eb15-43ed-4d91-bddc-611f707ae919.mp3.transcript.json
transcripts/denver_09d6e15f-a2b7-406b-ab38-9ff28

In [30]:
import os
import zipfile

mapping_extract_dir = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "denver_mapping"
)

os.makedirs(mapping_extract_dir, exist_ok=True)

with zipfile.ZipFile(denver_map_path, "r") as z:
    z.extractall(mapping_extract_dir)

print("Extracted to:")
print(mapping_extract_dir)

Extracted to:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping


In [31]:
for root, dirs, files in os.walk(mapping_extract_dir):
    for file in files:
        print(
            os.path.join(root, file)
        )

/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/denver_video_list.txt
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/transcripts/denver_0092aecb-4403-44ad-b9e4-a7a2733682af.mp3.transcript.json
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/transcripts/denver_009b1463-baf3-42f8-b3de-2820feea767c.mp3.transcript.json
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/transcripts/denver_01305102-03f9-4709-879f-40045397a390.mp3.transcript.json
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/transcripts/denver_0141a1aa-135d-41e6-a4c0-a97a6d93b717.mp3.transcript.json
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/transcripts/denver_05a9ae73-c724-4be9-8823-321b6ae9ee5d.mp3.transcript.json
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/transcripts/denver_05beac5d-717d-4b72-8d56-14b0c7909150.mp3.transcript.json
/content/drive/MyD

In [32]:
with zipfile.ZipFile(denver_map_path, "r") as z:
    print(z.namelist())

['denver_video_list.txt', 'transcripts/denver_0092aecb-4403-44ad-b9e4-a7a2733682af.mp3.transcript.json', 'transcripts/denver_009b1463-baf3-42f8-b3de-2820feea767c.mp3.transcript.json', 'transcripts/denver_01305102-03f9-4709-879f-40045397a390.mp3.transcript.json', 'transcripts/denver_0141a1aa-135d-41e6-a4c0-a97a6d93b717.mp3.transcript.json', 'transcripts/denver_05a9ae73-c724-4be9-8823-321b6ae9ee5d.mp3.transcript.json', 'transcripts/denver_05beac5d-717d-4b72-8d56-14b0c7909150.mp3.transcript.json', 'transcripts/denver_06505c2e-281d-462e-939c-ccb661ded293.mp3.transcript.json', 'transcripts/denver_0715097f-6484-4226-9433-ffae62bdf03e.mp3.transcript.json', 'transcripts/denver_07708b79-3284-474a-97c4-f4b1e566fbb7.mp3.transcript.json', 'transcripts/denver_077d2891-3e7c-4009-9cbb-7780290a5215.mp3.transcript.json', 'transcripts/denver_07a003c5-1b87-4ce9-a118-92c6d272f297.mp3.transcript.json', 'transcripts/denver_07c1eb15-43ed-4d91-bddc-611f707ae919.mp3.transcript.json', 'transcripts/denver_09d6e1

In [33]:
import zipfile
import os

mapping_extract_dir = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "denver_mapping"
)

os.makedirs(mapping_extract_dir, exist_ok=True)

with zipfile.ZipFile(denver_map_path, "r") as z:
    z.extract(
        "denver_video_list.txt",
        mapping_extract_dir
    )

video_list_path = os.path.join(
    mapping_extract_dir,
    "denver_video_list.txt"
)

print("Saved to:")
print(video_list_path)

Saved to:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/denver_mapping/denver_video_list.txt


In [34]:
with open(
    video_list_path,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    video_list_text = f.read()

print(video_list_text[:5000])

http://archive-media.granicus.com:443/OnDemand/denver/denver_97aa9adf-9e79-4f53-ab6c-7a2b690636b6.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_1ec5ccb0-e9d1-4bdb-9738-573eeaec6162.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_439d0265-91f3-4725-8773-8e69a8189cdb.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_cc90cc47-26fc-48cc-a566-5e94b27cbab8.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_af328ef4-ce40-4381-8aa4-6df4756ae665.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_cbdcc1d6-4203-4011-8e4e-0ed469a8f623.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_e42ed74f-d6e5-4e6d-9712-8adb44a78c13.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_82cb56a5-df16-4be5-9766-78f3eeb678f5.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_80e3c6ae-072c-4912-9252-460111ab95d5.mp4
http://archive-media.granicus.com:443/OnDemand/denver/denver_73c50112-33bd-47ad-a2

In [35]:
MEETING_ID = "DenverCityCouncil_05012017"

matches = [
    line for line in video_list_text.splitlines()
    if MEETING_ID.lower() in line.lower()
]

print("Number of matches:", len(matches))

for line in matches:
    print(line)

Number of matches: 0


In [36]:
import os
import json

MEETING_UID = "DenverCityCouncil_05012017_17-0161"

matches = []

for root, dirs, files in os.walk(mapping_extract_dir):

    for filename in files:

        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(root, filename)

        try:
            with open(
                filepath,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:
                data = json.load(f)

            # Convert the JSON to text so we can search
            # all fields safely.
            data_text = json.dumps(data)

            if MEETING_UID.lower() in data_text.lower():
                matches.append(filepath)

        except Exception as e:
            print("Could not read:", filepath)
            print("Reason:", e)

print("Number of matching JSON files:", len(matches))

for filepath in matches:
    print(filepath)

Number of matching JSON files: 0


In [37]:
SEARCH_TERMS = [
    "17-0161",
    "05012017",
    "2017-05-01",
    "05/01/2017"
]

for term in SEARCH_TERMS:

    matches = []

    for root, dirs, files in os.walk(mapping_extract_dir):

        for filename in files:

            if not filename.endswith(".json"):
                continue

            filepath = os.path.join(root, filename)

            try:
                with open(
                    filepath,
                    "r",
                    encoding="utf-8",
                    errors="ignore"
                ) as f:
                    text = f.read()

                if term.lower() in text.lower():
                    matches.append(filepath)

            except:
                pass

    print("\nSearch term:", term)
    print("Matches:", len(matches))

    for filepath in matches[:10]:
        print(filepath)


Search term: 17-0161
Matches: 0

Search term: 05012017
Matches: 0

Search term: 2017-05-01
Matches: 0

Search term: 05/01/2017
Matches: 0


In [38]:
%cd /content

!git clone https://github.com/YebowenHu/MeetingBank-utils.git

/content
Cloning into 'MeetingBank-utils'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 57 (delta 20), reused 26 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 20.50 KiB | 6.83 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [39]:
import os

utils_dir = "/content/MeetingBank-utils"

print("Repository exists:", os.path.exists(utils_dir))
print("\nFiles:")
for item in os.listdir(utils_dir):
    print(item)

Repository exists: True

Files:
README.md
utils
data
load_data.py
.git
ResultsEval.py


In [40]:
data_dir = os.path.join(
    utils_dir,
    "data"
)

print("Data directory exists:",
      os.path.exists(data_dir))

if os.path.exists(data_dir):
    for root, dirs, files in os.walk(data_dir):
        level = root.replace(data_dir, "").count(os.sep)

        if level > 2:
            continue

        print("\n", root)

        for file in files[:20]:
            print("   ", file)

Data directory exists: True

 /content/MeetingBank-utils/data
    README.md


In [41]:
import subprocess

result = subprocess.run(
    [
        "grep",
        "-R",
        "-l",
        "DenverCityCouncil_05012017",
        utils_dir
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

In [42]:
%cd /content

!git clone https://github.com/YebowenHu/MeetingBank-utils.git

/content
fatal: destination path 'MeetingBank-utils' already exists and is not an empty directory.


In [43]:
import os

utils_dir = "/content/MeetingBank-utils"

for root, dirs, files in os.walk(utils_dir):
    level = root.replace(utils_dir, "").count(os.sep)

    if level <= 2:
        print(root)
        for file in files[:20]:
            print("   ", file)

/content/MeetingBank-utils
    README.md
    load_data.py
    ResultsEval.py
/content/MeetingBank-utils/utils
    MoverScore.py
/content/MeetingBank-utils/data
    README.md
/content/MeetingBank-utils/.git
    HEAD
    packed-refs
    description
    config
    index
/content/MeetingBank-utils/.git/branches
/content/MeetingBank-utils/.git/hooks
    post-update.sample
    applypatch-msg.sample
    pre-commit.sample
    pre-rebase.sample
    pre-applypatch.sample
    update.sample
    commit-msg.sample
    pre-push.sample
    fsmonitor-watchman.sample
    pre-merge-commit.sample
    push-to-checkout.sample
    pre-receive.sample
    prepare-commit-msg.sample
/content/MeetingBank-utils/.git/refs
/content/MeetingBank-utils/.git/logs
    HEAD
/content/MeetingBank-utils/.git/objects
/content/MeetingBank-utils/.git/info
    exclude


In [44]:
import subprocess

result = subprocess.run(
    [
        "grep",
        "-R",
        "-l",
        "DenverCityCouncil_05012017",
        utils_dir
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

In [45]:
import requests

zenodo_api_url = "https://zenodo.org/api/records/7989108"

response = requests.get(zenodo_api_url)

print("Status code:", response.status_code)

data = response.json()

print("Record title:")
print(data["metadata"]["title"])

Status code: 200
Record title:
MeetingBank: A Benchmark Dataset for Meeting Summarization


In [46]:
for file_info in data["files"]:
    print(
        file_info["key"],
        "→",
        file_info.get("size", "unknown"),
        "bytes"
    )

MeetingBank.zip → 637072469 bytes


In [47]:
import requests
import os

zenodo_file_url = "https://zenodo.org/records/7989108/files/MeetingBank.zip"

zip_path = "/content/MeetingBank.zip"

print("Downloading MeetingBank.zip...")
print("Destination:", zip_path)

response = requests.get(
    zenodo_file_url,
    stream=True
)

response.raise_for_status()

total_size = int(
    response.headers.get("content-length", 0)
)

downloaded = 0

with open(zip_path, "wb") as f:

    for chunk in response.iter_content(
        chunk_size=1024 * 1024
    ):

        if chunk:

            f.write(chunk)
            downloaded += len(chunk)

            if total_size:
                percent = (
                    downloaded / total_size
                ) * 100

                print(
                    f"\rDownloaded: {percent:.1f}%",
                    end=""
                )

print("\nDownload complete.")

print(
    "File size:",
    os.path.getsize(zip_path) / (1024**2),
    "MB"
)

Destination: /content/MeetingBank.zip
Downloaded: 100.0%
Download complete.
File size: 607.5596513748169 MB


In [48]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:

    bad_file = z.testzip()

    if bad_file is None:
        print("✓ ZIP integrity check passed.")
    else:
        print("✗ Corrupted file:", bad_file)

✓ ZIP integrity check passed.


In [49]:
with zipfile.ZipFile(zip_path, "r") as z:

    all_files = z.namelist()

print("Total files:", len(all_files))

denver_files = [
    f for f in all_files
    if "/Denver/" in f or f.startswith("Denver/")
]

print("Denver-related entries:", len(denver_files))

for f in denver_files[:100]:
    print(f)

Total files: 1429
Denver-related entries: 404
Audio&Transcripts/Denver/
Audio&Transcripts/Denver/transcripts/
Audio&Transcripts/Denver/transcripts/denver_75d826bf-2f9d-495b-ae77-d77263f9491e.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_6f4dbe31-7984-482d-9ee9-bd00a2996685.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_19765381-101a-4539-a937-6ccc4e401931.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_6ae64b09-51ab-4ecf-bf57-ad4008cb3abc.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_5ed66562-5b35-4c54-946e-433eb7a9ce68.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_c799b208-cd4e-4996-9e6e-fd648ef8fbb2.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_fb8c74f8-c69f-4235-9f02-8427be52e0ce.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_b5022b9d-1f3e-4736-a762-9dc11d13607c.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_cb77d085-16c1-486e-8f53-eead8fe43a77.m

In [50]:
with zipfile.ZipFile(zip_path, "r") as z:

    transcript_files = [
        f for f in all_files
        if (
            "Audio&Transcripts/Denver/transcripts/"
            in f
            and f.endswith(".json")
        )
    ]

print(
    "Denver transcript JSON files:",
    len(transcript_files)
)

for f in transcript_files[:20]:
    print(f)

Denver transcript JSON files: 401
Audio&Transcripts/Denver/transcripts/denver_75d826bf-2f9d-495b-ae77-d77263f9491e.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_6f4dbe31-7984-482d-9ee9-bd00a2996685.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_19765381-101a-4539-a937-6ccc4e401931.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_6ae64b09-51ab-4ecf-bf57-ad4008cb3abc.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_5ed66562-5b35-4c54-946e-433eb7a9ce68.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_c799b208-cd4e-4996-9e6e-fd648ef8fbb2.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_fb8c74f8-c69f-4235-9f02-8427be52e0ce.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_b5022b9d-1f3e-4736-a762-9dc11d13607c.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_cb77d085-16c1-486e-8f53-eead8fe43a77.mp3.transcript.json
Audio&Transcripts/Denver/transcripts/denver_70cf4e2b-b77d

In [51]:
import json
import pprint

with zipfile.ZipFile(zip_path, "r") as z:

    sample_transcript_file = transcript_files[0]

    with z.open(sample_transcript_file) as f:
        transcript_data = json.load(f)

print("File:")
print(sample_transcript_file)

print("\nPython type:")
print(type(transcript_data))

print("\nStructure:")
pprint.pp(transcript_data, depth=3)

Streaming output truncated to the last 5000 lines.
               'offset': 32647499776,
               'speaker': 9,
               'nbest': [...]},
              {'duration': 232501248,
               'offset': 32936998912,
               'speaker': 9,
               'nbest': [...]},
              {'duration': 243900416,
               'offset': 33169500160,
               'speaker': 9,
               'nbest': [...]},
              {'duration': 160999424,
               'offset': 33427800064,
               'speaker': 9,
               'nbest': [...]},
              {'duration': 186601472,
               'offset': 33588899840,
               'speaker': 9,
               'nbest': [...]},
              {'duration': 74399744,
               'offset': 33781200896,
               'speaker': 9,
               'nbest': [...]},
              {'duration': 22200320,
               'offset': 33862199296,
               'speaker': 2,
               'nbest': [...]},
              {'duration': 197

In [52]:
print(transcript_data.keys())

dict_keys(['duration', 'engineVersion', 'language', 'segments', 'timestamp'])


In [53]:
print("Number of segments:",
      len(transcript_data["segments"]))

print("\nFirst segment:")
print(transcript_data["segments"][0])

Number of segments: 1460

First segment:
{'duration': 17500000, 'offset': 1200000, 'speaker': 0, 'nbest': [{'text': 'Station two stood here on this corner.', 'words': [{'confidence': 0.98, 'duration': 3600000, 'offset': 1200000, 'text': 'Station'}, {'confidence': 0.49, 'duration': 900000, 'offset': 4800000, 'text': 'two'}, {'confidence': 1.0, 'duration': 2600000, 'offset': 5800000, 'text': 'stood'}, {'confidence': 1.0, 'duration': 1800000, 'offset': 8400000, 'text': 'here'}, {'confidence': 1.0, 'duration': 1200000, 'offset': 10200000, 'text': 'on'}, {'confidence': 1.0, 'duration': 3300000, 'offset': 11400000, 'text': 'this'}, {'confidence': 1.0, 'duration': 4000000, 'offset': 14700000, 'text': 'corner'}]}]}


In [54]:
print("\nFirst 3 segments:")

for i, segment in enumerate(
    transcript_data["segments"][:3]
):
    print(f"\nSegment {i}:")
    print(segment)


First 3 segments:

Segment 0:
{'duration': 17500000, 'offset': 1200000, 'speaker': 0, 'nbest': [{'text': 'Station two stood here on this corner.', 'words': [{'confidence': 0.98, 'duration': 3600000, 'offset': 1200000, 'text': 'Station'}, {'confidence': 0.49, 'duration': 900000, 'offset': 4800000, 'text': 'two'}, {'confidence': 1.0, 'duration': 2600000, 'offset': 5800000, 'text': 'stood'}, {'confidence': 1.0, 'duration': 1800000, 'offset': 8400000, 'text': 'here'}, {'confidence': 1.0, 'duration': 1200000, 'offset': 10200000, 'text': 'on'}, {'confidence': 1.0, 'duration': 3300000, 'offset': 11400000, 'text': 'this'}, {'confidence': 1.0, 'duration': 4000000, 'offset': 14700000, 'text': 'corner'}]}]}

Segment 1:
{'duration': 30600000, 'offset': 18900000, 'speaker': 1, 'nbest': [{'text': "Y'all remember that? Now I want you.", 'words': [{'confidence': 0.59, 'duration': 2100000, 'offset': 18900000, 'text': "Y'all"}, {'confidence': 1.0, 'duration': 3300000, 'offset': 21000000, 'text': 'remem

In [55]:
print(transcript_data["segments"][0])

{'duration': 17500000, 'offset': 1200000, 'speaker': 0, 'nbest': [{'text': 'Station two stood here on this corner.', 'words': [{'confidence': 0.98, 'duration': 3600000, 'offset': 1200000, 'text': 'Station'}, {'confidence': 0.49, 'duration': 900000, 'offset': 4800000, 'text': 'two'}, {'confidence': 1.0, 'duration': 2600000, 'offset': 5800000, 'text': 'stood'}, {'confidence': 1.0, 'duration': 1800000, 'offset': 8400000, 'text': 'here'}, {'confidence': 1.0, 'duration': 1200000, 'offset': 10200000, 'text': 'on'}, {'confidence': 1.0, 'duration': 3300000, 'offset': 11400000, 'text': 'this'}, {'confidence': 1.0, 'duration': 4000000, 'offset': 14700000, 'text': 'corner'}]}]}


In [57]:
print("Total lines:", len(lines))

for i, line in enumerate(lines[:10]):
    print(f"{i}: {repr(line)}")

NameError: name 'lines' is not defined

In [ ]:
print("Last 10 lines:")

for i, line in enumerate(lines[-10:], start=len(lines)-10):
    print(f"{i}: {repr(line)}")

In [58]:
# Read the Denver video list again

with open(
    video_list_path,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    lines = f.read().splitlines()

print("Total lines:", len(lines))

print("\nFirst 10 lines:")
for i, line in enumerate(lines[:10]):
    print(f"{i}: {repr(line)}")

Total lines: 408

First 10 lines:
0: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_97aa9adf-9e79-4f53-ab6c-7a2b690636b6.mp4'
1: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_1ec5ccb0-e9d1-4bdb-9738-573eeaec6162.mp4'
2: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_439d0265-91f3-4725-8773-8e69a8189cdb.mp4'
3: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_cc90cc47-26fc-48cc-a566-5e94b27cbab8.mp4'
4: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_af328ef4-ce40-4381-8aa4-6df4756ae665.mp4'
5: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_cbdcc1d6-4203-4011-8e4e-0ed469a8f623.mp4'
6: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_e42ed74f-d6e5-4e6d-9712-8adb44a78c13.mp4'
7: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_82cb56a5-df16-4be5-9766-78f3eeb678f5.mp4'
8: 'http://archive-media.granicus.com:443/OnDemand/denver/denver_80e3c6ae-072c-4912-9252-460111ab95d5.mp4'
9: 

In [59]:
import os
import json
import zipfile
import re

REFERENCE_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "references",
    "0_reference_transcript.txt"
)

with open(
    REFERENCE_PATH,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    reference_text = f.read()

print("Reference transcript characters:", len(reference_text))
print("\nBeginning of reference transcript:")
print(reference_text[:1000])

Reference transcript characters: 152314

Beginning of reference transcript:
Please refrain from profane or obscene speech. Direct your comments to council as a whole and refrain from individual or personal attacks. Councilwoman Gilmore, will you please put Council Bill 161 on the floor? Yes, President Brooks, I move that council bill 161 as amended, be placed upon final consideration and do pass. It has been moved and seconded. Councilman. Clerk, go ahead and offer your motion to further amend. Thank you, Mr. President. I move that council bill 161 as amended be further amended in the following particulars on page five, line one, strike 13 .1.9 and replace with 13 .1. ten on page five. Line three strike 13 1.9.3 and replace with 13 one dot and four. Add a new section to the bill which reads as follows Section six A except as otherwise provided in Section six B of this ordinance. With respect to certain site development plan applications, the amendments to the Denver Zoning Code adopted

In [61]:
def extract_json_transcript(data):
    texts = []

    for segment in data.get("segments", []):

        nbest = segment.get("nbest", [])

        if nbest:
            text = nbest[0].get("text", "")
            texts.append(text)

    return " ".join(texts)

In [62]:
def normalize_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [63]:
reference_norm = normalize_text(reference_text)

reference_words = reference_norm.split()

print("Reference words:", len(reference_words))

fingerprint = " ".join(
    reference_words[:50]
)

print("\nReference fingerprint:")
print(fingerprint)

Reference words: 28503

Reference fingerprint:
please refrain from profane or obscene speech direct your comments to council as a whole and refrain from individual or personal attacks councilwoman gilmore will you please put council bill 161 on the floor yes president brooks i move that council bill 161 as amended be placed upon final consideration


In [64]:
candidates = []

with zipfile.ZipFile(zip_path, "r") as z:

    for transcript_file in transcript_files:

        try:
            with z.open(transcript_file) as f:
                data = json.load(f)

            transcript = extract_json_transcript(data)
            transcript_norm = normalize_text(transcript)

            # Count how many reference fingerprint words
            # appear in the candidate transcript.
            candidate_words = set(
                transcript_norm.split()
            )

            fingerprint_words = set(
                fingerprint.split()
            )

            overlap = len(
                candidate_words.intersection(
                    fingerprint_words
                )
            )

            candidates.append(
                (
                    overlap,
                    transcript_file
                )
            )

        except Exception as e:
            print(
                "Error:",
                transcript_file,
                e
            )

candidates.sort(
    reverse=True
)

print("Top candidates:")
for score, filename in candidates[:20]:
    print(score, filename)

Top candidates:
41 Audio&Transcripts/Denver/transcripts/denver_c3d0f6da-545a-47de-87ed-fb1a4cff049b.mp3.transcript.json
41 Audio&Transcripts/Denver/transcripts/denver_8b23b983-62ce-4f7d-ad0b-c939781b75eb.mp3.transcript.json
41 Audio&Transcripts/Denver/transcripts/denver_7d84bc3c-1295-46e5-a823-30c648eb2308.mp3.transcript.json
41 Audio&Transcripts/Denver/transcripts/denver_202e51c5-2ed9-458b-91c6-6bccc2012578.mp3.transcript.json
40 Audio&Transcripts/Denver/transcripts/denver_f99da1ba-3f99-42a3-8a97-094276aaf6fc.mp3.transcript.json
40 Audio&Transcripts/Denver/transcripts/denver_f31c3c0f-d0e2-4c36-a66b-a31fdce5cd6d.mp3.transcript.json
40 Audio&Transcripts/Denver/transcripts/denver_f1ac6bf0-4b5f-4725-af32-d0f1cb0f7923.mp3.transcript.json
40 Audio&Transcripts/Denver/transcripts/denver_ee673d7c-0141-4607-8060-9380a13fd6c7.mp3.transcript.json
40 Audio&Transcripts/Denver/transcripts/denver_e789d292-f9e2-4547-b1b8-73b5fce8f3c8.mp3.transcript.json
40 Audio&Transcripts/Denver/transcripts/denver_e

In [65]:
print("Top candidates:")

Top candidates:


In [66]:
from difflib import SequenceMatcher
import json
import zipfile
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

reference_norm = normalize_text(reference_text)

# Use a reasonably large section of the reference transcript.
reference_sample = reference_norm[:20000]

verified_candidates = []

with zipfile.ZipFile(zip_path, "r") as z:

    for overlap, transcript_file in candidates[:20]:

        try:
            with z.open(transcript_file) as f:
                data = json.load(f)

            candidate_text = extract_json_transcript(data)
            candidate_norm = normalize_text(candidate_text)

            # Compare the beginning of both transcripts.
            candidate_sample = candidate_norm[:20000]

            similarity = SequenceMatcher(
                None,
                reference_sample,
                candidate_sample
            ).ratio()

            verified_candidates.append(
                (
                    similarity,
                    transcript_file,
                    data.get("duration"),
                    len(data.get("segments", []))
                )
            )

        except Exception as e:
            print("Error:", transcript_file, e)

verified_candidates.sort(
    reverse=True,
    key=lambda x: x[0]
)

print("Verified candidates:\n")

for similarity, filename, duration, segments in verified_candidates:
    print(
        f"Similarity: {similarity:.4f} | "
        f"Duration: {duration} | "
        f"Segments: {segments}"
    )
    print(filename)
    print()

Verified candidates:

Similarity: 0.0126 | Duration: 80669999104 | Segments: 570
Audio&Transcripts/Denver/transcripts/denver_d45faee2-9a61-49f6-a5f0-711ff708947c.mp3.transcript.json

Similarity: 0.0111 | Duration: 68730003456 | Segments: 537
Audio&Transcripts/Denver/transcripts/denver_8b23b983-62ce-4f7d-ad0b-c939781b75eb.mp3.transcript.json

Similarity: 0.0092 | Duration: 67799998464 | Segments: 486
Audio&Transcripts/Denver/transcripts/denver_c3d0f6da-545a-47de-87ed-fb1a4cff049b.mp3.transcript.json

Similarity: 0.0072 | Duration: 147119996928 | Segments: 1252
Audio&Transcripts/Denver/transcripts/denver_f31c3c0f-d0e2-4c36-a66b-a31fdce5cd6d.mp3.transcript.json

Similarity: 0.0070 | Duration: 84020002816 | Segments: 676
Audio&Transcripts/Denver/transcripts/denver_e6e87e07-4983-4f1a-b242-26b2d3755b7a.mp3.transcript.json

Similarity: 0.0064 | Duration: 165889998848 | Segments: 1280
Audio&Transcripts/Denver/transcripts/denver_e789d292-f9e2-4547-b1b8-73b5fce8f3c8.mp3.transcript.json

Similari

In [67]:
print("Reference transcript length:",
      len(reference_text))

print("\nReference transcript:")
print(reference_text[:3000])

Reference transcript length: 152314

Reference transcript:
Please refrain from profane or obscene speech. Direct your comments to council as a whole and refrain from individual or personal attacks. Councilwoman Gilmore, will you please put Council Bill 161 on the floor? Yes, President Brooks, I move that council bill 161 as amended, be placed upon final consideration and do pass. It has been moved and seconded. Councilman. Clerk, go ahead and offer your motion to further amend. Thank you, Mr. President. I move that council bill 161 as amended be further amended in the following particulars on page five, line one, strike 13 .1.9 and replace with 13 .1. ten on page five. Line three strike 13 1.9.3 and replace with 13 one dot and four. Add a new section to the bill which reads as follows Section six A except as otherwise provided in Section six B of this ordinance. With respect to certain site development plan applications, the amendments to the Denver Zoning Code adopted by this ordinanc

In [68]:
reference_words = normalize_text(reference_text).split()

print("\nNumber of reference words:",
      len(reference_words))


Number of reference words: 28503


In [69]:
# Create several phrases from the reference transcript.
# Each phrase contains 12 consecutive words.

reference_norm = normalize_text(reference_text)
reference_words = reference_norm.split()

phrases = []

for i in range(
    0,
    max(1, len(reference_words) - 12),
    20
):
    phrase = " ".join(
        reference_words[i:i+12]
    )

    if len(phrase.split()) >= 12:
        phrases.append(phrase)

print("Number of phrases:", len(phrases))

print("\nFirst 10 phrases:")
for phrase in phrases[:10]:
    print("-", phrase)

Number of phrases: 1425

First 10 phrases:
- please refrain from profane or obscene speech direct your comments to council
- personal attacks councilwoman gilmore will you please put council bill 161 on
- council bill 161 as amended be placed upon final consideration and do
- clerk go ahead and offer your motion to further amend thank you
- as amended be further amended in the following particulars on page five
- with 13 1 ten on page five line three strike 13 1
- and four add a new section to the bill which reads as
- in section six b of this ordinance with respect to certain site
- zoning code adopted by this ordinance take effect on may five 2017
- if requested by an applicant a pending formal site development plan application


In [70]:
phrase_matches = []

with zipfile.ZipFile(zip_path, "r") as z:

    for transcript_file in transcript_files:

        try:
            with z.open(transcript_file) as f:
                data = json.load(f)

            candidate_text = extract_json_transcript(data)
            candidate_norm = normalize_text(candidate_text)

            matched_phrases = []

            for phrase in phrases:

                if phrase in candidate_norm:
                    matched_phrases.append(phrase)

            if matched_phrases:

                phrase_matches.append({
                    "file": transcript_file,
                    "matches": len(matched_phrases),
                    "duration": data.get("duration"),
                    "segments": len(
                        data.get("segments", [])
                    )
                })

        except Exception as e:
            print(
                "Error reading:",
                transcript_file,
                e
            )

phrase_matches.sort(
    key=lambda x: x["matches"],
    reverse=True
)

print("Candidates with exact phrase matches:",
      len(phrase_matches))

for item in phrase_matches[:20]:
    print(
        "\nMatches:",
        item["matches"],
        "| Duration:",
        item["duration"],
        "| Segments:",
        item["segments"]
    )
    print(item["file"])

Candidates with exact phrase matches: 139

Matches: 1425 | Duration: 142550007808 | Segments: 1029
Audio&Transcripts/Denver/transcripts/denver_202e51c5-2ed9-458b-91c6-6bccc2012578.mp3.transcript.json

Matches: 1 | Duration: 130039996416 | Segments: 1096
Audio&Transcripts/Denver/transcripts/denver_6ae64b09-51ab-4ecf-bf57-ad4008cb3abc.mp3.transcript.json

Matches: 1 | Duration: 67090001920 | Segments: 424
Audio&Transcripts/Denver/transcripts/denver_5ed66562-5b35-4c54-946e-433eb7a9ce68.mp3.transcript.json

Matches: 1 | Duration: 65550000128 | Segments: 443
Audio&Transcripts/Denver/transcripts/denver_b5022b9d-1f3e-4736-a762-9dc11d13607c.mp3.transcript.json

Matches: 1 | Duration: 54200000512 | Segments: 410
Audio&Transcripts/Denver/transcripts/denver_a696e8eb-7015-43e1-9499-541d178c2b8c.mp3.transcript.json

Matches: 1 | Duration: 180539998208 | Segments: 1541
Audio&Transcripts/Denver/transcripts/denver_9346cc84-35c3-40ea-9a20-81399a31d3b5.mp3.transcript.json

Matches: 1 | Duration: 5353999

In [71]:
TARGET_JSON = (
    "Audio&Transcripts/Denver/transcripts/"
    "denver_202e51c5-2ed9-458b-91c6-6bccc2012578.mp3.transcript.json"
)

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(TARGET_JSON) as f:
        target_data = json.load(f)

target_text = extract_json_transcript(target_data)

print("Reference transcript characters:", len(reference_text))
print("Candidate transcript characters:", len(target_text))

print("\nReference beginning:")
print(reference_text[:1000])

print("\nCandidate beginning:")
print(target_text[:1000])

Reference transcript characters: 152314
Candidate transcript characters: 191181

Reference beginning:
Please refrain from profane or obscene speech. Direct your comments to council as a whole and refrain from individual or personal attacks. Councilwoman Gilmore, will you please put Council Bill 161 on the floor? Yes, President Brooks, I move that council bill 161 as amended, be placed upon final consideration and do pass. It has been moved and seconded. Councilman. Clerk, go ahead and offer your motion to further amend. Thank you, Mr. President. I move that council bill 161 as amended be further amended in the following particulars on page five, line one, strike 13 .1.9 and replace with 13 .1. ten on page five. Line three strike 13 1.9.3 and replace with 13 one dot and four. Add a new section to the bill which reads as follows Section six A except as otherwise provided in Section six B of this ordinance. With respect to certain site development plan applications, the amendments to the 

In [72]:
reference_norm = normalize_text(reference_text)
target_norm = normalize_text(target_text)

reference_words = reference_norm.split()

print("Reference word count:", len(reference_words))

Reference word count: 28503


In [73]:
# Take a 30-word phrase from the middle of the reference
middle = len(reference_words) // 2

middle_phrase = " ".join(
    reference_words[middle:middle + 30]
)

print("Middle phrase:")
print(middle_phrase)

print("\nExact match in candidate:",
      middle_phrase in target_norm)

Middle phrase:
underlying intent that this had to be something pretty extraordinary to get more than 50 we don t have that kind of direction here it s do you get the

Exact match in candidate: True


In [74]:
sections = {
    "beginning": 0,
    "25_percent": len(reference_words) // 4,
    "middle": len(reference_words) // 2,
    "75_percent": (len(reference_words) * 3) // 4
}

for name, start in sections.items():

    phrase = " ".join(
        reference_words[start:start + 30]
    )

    print(
        name,
        "→",
        phrase in target_norm
    )

beginning → True
25_percent → True
middle → True
75_percent → True


We are done with dataset mapping. 🎯

Next: download only this meeting

We know the corresponding video URL from denver_video_list.txt follows the same UUID pattern.

In [75]:
import os
import requests

VIDEO_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

VIDEO_URL = (
    f"http://archive-media.granicus.com:443/"
    f"OnDemand/denver/"
    f"denver_{VIDEO_UUID}.mp4"
)

RAW_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw"
)

os.makedirs(RAW_DIR, exist_ok=True)

VIDEO_PATH = os.path.join(
    RAW_DIR,
    f"DenverCityCouncil_05012017_{VIDEO_UUID}.mp4"
)

print("Video URL:")
print(VIDEO_URL)

print("\nSaving to:")
print(VIDEO_PATH)

Video URL:
http://archive-media.granicus.com:443/OnDemand/denver/denver_202e51c5-2ed9-458b-91c6-6bccc2012578.mp4

Saving to:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/DenverCityCouncil_05012017_202e51c5-2ed9-458b-91c6-6bccc2012578.mp4


In [76]:
response = requests.head(
    VIDEO_URL,
    allow_redirects=True,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print(
    "Content length:",
    response.headers.get("Content-Length")
)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [77]:
import requests

VIDEO_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

VIDEO_URL = (
    f"http://archive-media.granicus.com:443/"
    f"OnDemand/denver/"
    f"denver_{VIDEO_UUID}.mp4"
)

headers = {
    "Range": "bytes=0-1023",
    "User-Agent": "Mozilla/5.0"
}

try:
    response = requests.get(
        VIDEO_URL,
        headers=headers,
        timeout=60,
        stream=True
    )

    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    print("Content range:", response.headers.get("Content-Range"))
    print("Content length:", response.headers.get("Content-Length"))

except Exception as e:
    print("Connection error:")
    print(type(e).__name__, e)

Connection error:
ConnectionError ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [78]:
import zipfile

TARGET_MP3 = (
    "denver_202e51c5-2ed9-458b-91c6-6bccc2012578.mp3"
)

with zipfile.ZipFile(zip_path, "r") as z:

    mp3_matches = [
        f for f in z.namelist()
        if f.endswith(TARGET_MP3)
    ]

print("Matches found:", len(mp3_matches))

for f in mp3_matches:
    print(f)

Matches found: 0


In [79]:
import zipfile

TARGET_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

with zipfile.ZipFile(zip_path, "r") as z:

    matches = [
        f for f in z.namelist()
        if TARGET_UUID.lower() in f.lower()
    ]

print("UUID matches:", len(matches))

for f in matches:
    print(f)

UUID matches: 1
Audio&Transcripts/Denver/transcripts/denver_202e51c5-2ed9-458b-91c6-6bccc2012578.mp3.transcript.json


So the corresponding MP3 is not directly stored in MeetingBank.zip's top-level file list; it's inside one of the nested Denver MP3 ZIP archives.

We can now locate it without downloading/extracting all 21 archives.

Step 1 — List the Denver MP3 archives

In [80]:
with zipfile.ZipFile(zip_path, "r") as z:
    denver_archives = [
        f for f in z.namelist()
        if f.startswith("Audio&Transcripts/Denver/mp3/")
        and f.endswith(".zip")
    ]

print("Denver MP3 archives:", len(denver_archives))

for f in denver_archives:
    print(f)

Denver MP3 archives: 0


In [81]:
TARGET_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

archive_match = None
mp3_match = None

with zipfile.ZipFile(zip_path, "r") as outer_zip:

    for archive_path in denver_archives:

        print("Checking:", archive_path)

        # Read the nested ZIP into memory
        nested_zip_bytes = outer_zip.read(archive_path)

        import io

        with zipfile.ZipFile(
            io.BytesIO(nested_zip_bytes),
            "r"
        ) as inner_zip:

            for filename in inner_zip.namelist():

                if TARGET_UUID.lower() in filename.lower():

                    archive_match = archive_path
                    mp3_match = filename

                    print("\nFOUND!")
                    print("Archive:", archive_match)
                    print("File:", mp3_match)

                    break

        if mp3_match is not None:
            break

print("\nFinal result:")
print("Archive:", archive_match)
print("MP3:", mp3_match)


Final result:
Archive: None
MP3: None


In [83]:
import io
import zipfile

with zipfile.ZipFile(zip_path, "r") as outer_zip:

    # Take the first Denver MP3 archive
    test_archive = denver_archives[0]

    print("Inspecting:")
    print(test_archive)

    nested_bytes = outer_zip.read(test_archive)

    with zipfile.ZipFile(
        io.BytesIO(nested_bytes),
        "r"
    ) as inner_zip:

        inner_files = inner_zip.namelist()

        print("\nNumber of files:", len(inner_files))

        print("\nFirst 30 files:")
        for f in inner_files[:30]:
            print(repr(f))

IndexError: list index out of range

In [84]:
import os
import io
import zipfile

# --------------------------------------------------
# 1. Locate MeetingBank.zip
# --------------------------------------------------

zip_path = "/content/MeetingBank.zip"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        "MeetingBank.zip is not present at /content/MeetingBank.zip"
    )

print("MeetingBank.zip found.")
print(
    "Size:",
    round(os.path.getsize(zip_path) / (1024**2), 2),
    "MB"
)


# --------------------------------------------------
# 2. Find Denver MP3 archives inside MeetingBank.zip
# --------------------------------------------------

with zipfile.ZipFile(zip_path, "r") as outer_zip:

    all_files = outer_zip.namelist()

    denver_archives = [
        f for f in all_files
        if (
            "Audio&Transcripts/Denver/mp3/" in f
            and f.lower().endswith(".zip")
        )
    ]

print("\nDenver MP3 archives found:", len(denver_archives))

for archive in denver_archives:
    print(archive)


# --------------------------------------------------
# 3. If archives were found, inspect the first one
# --------------------------------------------------

if not denver_archives:
    print("\nNo nested Denver MP3 archives were found.")
else:

    test_archive = denver_archives[0]

    print("\nInspecting:")
    print(test_archive)

    with zipfile.ZipFile(zip_path, "r") as outer_zip:

        nested_bytes = outer_zip.read(test_archive)

    with zipfile.ZipFile(
        io.BytesIO(nested_bytes),
        "r"
    ) as inner_zip:

        inner_files = inner_zip.namelist()

        print(
            "\nNumber of files inside archive:",
            len(inner_files)
        )

        print("\nFirst 30 files:")

        for f in inner_files[:30]:
            print(repr(f))

MeetingBank.zip found.
Size: 607.56 MB

Denver MP3 archives found: 0

No nested Denver MP3 archives were found.


In [85]:
from huggingface_hub import list_repo_files

AUDIO_REPO = "huuuyeah/MeetingBank_Audio"

audio_files = list_repo_files(
    repo_id=AUDIO_REPO,
    repo_type="dataset"
)

denver_audio_archives = [
    f for f in audio_files
    if f.startswith("Denver/mp3/")
    and f.endswith(".zip")
]

print("Denver audio archives:")
for f in denver_audio_archives:
    print(f)

Denver audio archives:
Denver/mp3/Denver-1.zip
Denver/mp3/Denver-10.zip
Denver/mp3/Denver-11.zip
Denver/mp3/Denver-12.zip
Denver/mp3/Denver-13.zip
Denver/mp3/Denver-14.zip
Denver/mp3/Denver-15.zip
Denver/mp3/Denver-16.zip
Denver/mp3/Denver-17.zip
Denver/mp3/Denver-18.zip
Denver/mp3/Denver-19.zip
Denver/mp3/Denver-2.zip
Denver/mp3/Denver-20.zip
Denver/mp3/Denver-21.zip
Denver/mp3/Denver-3.zip
Denver/mp3/Denver-4.zip
Denver/mp3/Denver-5.zip
Denver/mp3/Denver-6.zip
Denver/mp3/Denver-7.zip
Denver/mp3/Denver-8.zip
Denver/mp3/Denver-9.zip


In [86]:
from huggingface_hub import hf_hub_download
import zipfile
import os
import io

TARGET_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

AUDIO_REPO = "huuuyeah/MeetingBank_Audio"

denver_archives = [
    f for f in audio_files
    if f.startswith("Denver/mp3/")
    and f.endswith(".zip")
]

print("Searching", len(denver_archives), "Denver archives...")
print()

found_archive = None
found_file = None

for archive in denver_archives:

    print("Checking:", archive)

    try:
        # Download one archive to Colab's cache.
        local_archive = hf_hub_download(
            repo_id=AUDIO_REPO,
            filename=archive,
            repo_type="dataset"
        )

        with zipfile.ZipFile(
            local_archive,
            "r"
        ) as z:

            matches = [
                f for f in z.namelist()
                if TARGET_UUID.lower() in f.lower()
            ]

            if matches:
                found_archive = archive
                found_file = matches[0]

                print("\nFOUND!")
                print("Archive:", found_archive)
                print("File:", found_file)
                break

    except Exception as e:
        print("Could not inspect:", archive)
        print(type(e).__name__, e)

print("\n========== RESULT ==========")
print("Archive:", found_archive)
print("File:", found_file)

Searching 21 Denver archives...

Checking: Denver/mp3/Denver-1.zip


Denver/mp3/Denver-1.zip: reconstructing file:   0%|          |  0.00B / 2.19GB            

Denver/mp3/Denver-1.zip: downloading bytes:           |  0.00B            

Checking: Denver/mp3/Denver-10.zip


Denver/mp3/Denver-10.zip: reconstructing file:   0%|          |  0.00B / 3.39GB            

Denver/mp3/Denver-10.zip: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [1]:
from huggingface_hub import list_repo_tree

TARGET_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

results = []

for item in list_repo_tree(
    "huuuyeah/MeetingBank_Audio",
    path="Denver/mp3",
    repo_type="dataset",
    recursive=True
):
    item_path = getattr(item, "path", "")

    if TARGET_UUID.lower() in item_path.lower():
        results.append(item_path)

print("Matches:", len(results))

for r in results:
    print(r)

TypeError: HfApi.list_repo_tree() got an unexpected keyword argument 'path'

In [2]:
from huggingface_hub import HfApi

TARGET_UUID = "202e51c5-2ed9-458b-91c6-6bccc2012578"

api = HfApi()

results = []

for item in api.list_repo_tree(
    repo_id="huuuyeah/MeetingBank_Audio",
    repo_type="dataset",
    path_in_repo="Denver/mp3",
    recursive=True
):
    item_path = getattr(item, "path", "")

    if TARGET_UUID.lower() in item_path.lower():
        results.append(item_path)

print("Matches:", len(results))

for r in results:
    print(r)

Matches: 0
